<a href="https://colab.research.google.com/github/karthikkan-otamiser/Automation/blob/main/HubspotxRank_Updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install papermill jupyter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.9/88.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.8 MB/s eta 0:00:00


In [3]:
import re
import requests
from google.colab import files
from datetime import datetime, date, timedelta
import pandas as pd
import gspread
from gspread.exceptions import WorksheetNotFound
from datetime import datetime
from urllib.parse import urlparse
from urllib.parse import urlparse, parse_qs
import math
from gspread.utils import rowcol_to_a1
import time
from gspread.exceptions import APIError

In [4]:
uploaded = files.upload()

SERVICE_ACCOUNT_FILE = "revnest-sheet-sync-7df3dbadd2b3.json"
SPREADSHEET_NAME = "Hubspot Ranking List"

TAB_IDS = "IDs"
TAB_LISTINGS = "Listings"
TAB_RAW = "Data Raw Updated"
TAB_REVIEWS = "Reviews Raw Updated"

API_KEY = "f0e3a3e4-6207-438c-92d3-4a2f4a3907df"
MARKET_SIZE = 270

URL_ONBOARD = "https://api.price-wise.io/bnb/input_parameters"
URL_RANK = "https://api.price-wise.io/bnb/ranking"
URL_OFFBOARD = "https://api.price-wise.io/bnb/input_parameters"

HEADERS_POST = {"accept":"application/json","Content-Type":"application/json","X-API-Key":API_KEY}
HEADERS_GET  = {"accept":"application/json","X-API-Key":API_KEY}
HEADERS_DEL  = {"accept":"application/json","X-API-Key":API_KEY}

Saving revnest-sheet-sync-7df3dbadd2b3.json to revnest-sheet-sync-7df3dbadd2b3 (1).json


In [5]:
def safe_batch_update(ws, updates, chunk_size=100, max_retries=6):
    if not updates:
        return

    for i in range(0, len(updates), chunk_size):
        chunk = updates[i:i + chunk_size]

        for attempt in range(max_retries):
            try:
                ws.batch_update(chunk)
                break
            except APIError as e:
                msg = str(e)

                if "429" in msg and attempt < max_retries - 1:
                    sleep_s = 2 ** attempt   # 1, 2, 4, 8, 16, 32
                    print(f"⏳ Sheets quota hit (429). Retrying in {sleep_s}s...")
                    time.sleep(sleep_s)
                else:
                    raise

In [6]:
def safe_append_rows(ws, rows, value_input_option="USER_ENTERED", table_range="A1", max_retries=6):
    if not rows:
        return

    for attempt in range(max_retries):
        try:
            ws.append_rows(
                rows,
                value_input_option=value_input_option,
                table_range=table_range
            )
            return
        except APIError as e:
            msg = str(e)
            if "429" in msg and attempt < max_retries - 1:
                sleep_s = 2 ** attempt
                print(f"⏳ Sheets append quota hit (429). Retrying in {sleep_s}s...")
                time.sleep(sleep_s)
            else:
                raise

In [7]:
def now_iso():
    return datetime.now().isoformat(timespec="seconds")

def parse_airbnb_id(url: str):
    if not url:
        return None
    # Standard guest-facing URL: /rooms/123456
    m = re.search(r"/rooms/(\d+)", url)
    if m:
        return m.group(1)
    # Host editor URL: /hosting/listings/editor/123456/...
    m2 = re.search(r"/hosting/listings/(?:editor/)?(\d+)", url)
    if m2:
        return m2.group(1)
    # Fallback: first long number in URL
    m3 = re.search(r"(\d{6,})", url)
    return m3.group(1) if m3 else None

def parse_guest_count(max_guests_str: str):
    """Extract number from 'X guests maximum' format"""
    if not max_guests_str:
        return ""
    m = re.search(r"(\d+)", max_guests_str)
    return m.group(1) if m else ""

def ensure_worksheet(ss, title, headers):
    try:
        ws = ss.worksheet(title)
    except WorksheetNotFound:
        ws = ss.add_worksheet(title=title, rows="3000", cols=str(max(12, len(headers)+2)))
        ws.append_row(headers)
        return ws

    existing = ws.row_values(1)
    if not existing:
        ws.append_row(headers)
    else:
        new = existing[:]
        for h in headers:
            if h not in new:
                new.append(h)
        if new != existing:
            ws.update("1:1", [new])
    return ws

def provider_onboard(listing_id: str):
    payload = {
        "target_listing_id": str(listing_id),
        "adults": 0,
        "region": "",
        "latitude": 0,
        "longitude": 0,
        "calibrated_delta_270": 0,
        "calibrated_delta_500": 0,
        "calibrated_delta_1000": 0
    }
    r = requests.post(URL_ONBOARD, headers=HEADERS_POST, json=payload, timeout=30)
    return r.status_code, r.text

def provider_offboard(listing_id: str):
    r = requests.delete(f"{URL_OFFBOARD}?target_listing_id={listing_id}", headers=HEADERS_DEL, timeout=30)
    return r.status_code, r.text

def provider_get_rank(listing_id: str, rank_date: str):
    r = requests.get(
        f"{URL_RANK}?target_listing_id={listing_id}&market_size={MARKET_SIZE}&date={rank_date}",
        headers=HEADERS_GET,
        timeout=30
    )
    if r.status_code != 200:
        return r.status_code, r.text
    return 200, r.json()

def provider_get_content(listing_id: str):
    """Fetch listing content (reviews, ratings, guest info)"""
    r = requests.get(
        f"https://api.price-wise.io/bnb/content?target_listing_id={listing_id}",
        headers=HEADERS_GET,
        timeout=30
    )
    if r.status_code != 200:
        return r.status_code, r.text
    return 200, r.json()

def to_date_str(x):
    if isinstance(x, str):
        return x.strip()[:10]
    return str(x)[:10]

def parse_airbnb_id_strict(url: str) -> str | None:
    if not url:
        return None

    url = url.strip()

    # Reject plain numbers (forces URL)
    if re.fullmatch(r"\d{5,}", url):
        return None

    try:
        u = urlparse(url)
    except Exception:
        return None

    host = (u.netloc or "").lower()

    # Must contain "airbnb" in the domain
    if "airbnb" not in host:
        return None

    # /rooms/ URL
    m = re.search(r"/rooms/(\d{5,})", u.path)
    if m:
        return m.group(1)

    # Host editor URL: /hosting/listings/editor/123456/...
    m2 = re.search(r"/hosting/listings/(?:editor/)?(\d{5,})", u.path)
    if m2:
        return m2.group(1)

    return None

LABELS = {
    "booking": "Booking.com",
    "number": "Number",
    "airbnb_non_rooms": "Airbnb link (not /rooms/)",
    "other_link": "Other link",
    "text": "Text",
}

def classify_input(x: str) -> str:
    x = (x or "").strip()
    if not x:
        return "text"
    if x.isdigit():
        return "number"
    try:
        host = urlparse(x).netloc.lower()
        path = (urlparse(x).path or "").lower()
    except Exception:
        return "text"

    if "booking" in host:
        return "booking"
    if "airbnb" in host:
        # Valid if /rooms/ or /hosting/listings/
        if "/rooms/" in path or "/hosting/listings/" in path:
            return "airbnb"
        return "airbnb_non_rooms"
    if host:
        return "other_link"
    return "text"

def build_csm_note(valid_count: int, invalid_count: int, invalid_types: dict, status: str,
                   added_count: int, existed_count: int) -> str:
    if status == "YES":
        parts = [f"{valid_count} valid Airbnb link(s)."]
    elif status == "INVALID_URL":
        parts = [f"{invalid_count} invalid input(s)."]
    elif status == "PARTIAL":
        parts = [f"{valid_count} valid Airbnb link(s), {invalid_count} invalid input(s)."]
    elif status == "ERROR":
        return "Script error while processing this row."

    if status in {"INVALID_URL", "PARTIAL"} and invalid_types:
        details = ", ".join([f"{LABELS.get(k, k)}: {v}" for k, v in sorted(invalid_types.items())])
        parts.append(f"({details}).")

    if status in {"YES", "PARTIAL"}:
        if added_count or existed_count:
            parts.append(f"Added: {added_count}, already existed: {existed_count}.")

    return " ".join(parts).strip()
def calculate_bbox_area_km2(ne_lat, ne_lng, sw_lat, sw_lng):
    if None in (ne_lat, ne_lng, sw_lat, sw_lng):
        return None
    R = 6371
    area = (
        R ** 2
        * abs(math.sin(math.radians(ne_lat)) - math.sin(math.radians(sw_lat)))
        * abs(math.radians(ne_lng) - math.radians(sw_lng))
    )
    return area

def classify_market_density(density):
    if density is None:
        return None
    elif density < 10:
        return "Low Density [0-10)"
    elif density < 100:
        return "Moderate Density [10-100)"
    elif density < 500:
        return "Urban Neighborhood [100-500)"
    elif density < 1500:
        return "Dense Urban [500-1500)"
    else:
        return "Hyper Dense (>=1500)"

def get_market_density(listing_id: str):
    """Fetch market density from input_parameters/listing endpoint"""
    try:
        r = requests.get(
            f"https://api.price-wise.io/bnb/input_parameters/listing?target_listing_id={listing_id}",
            headers=HEADERS_GET,
            timeout=30
        )
        if r.status_code != 200:
            print(f"   ⚠️ Density API status {r.status_code} for {listing_id}")
            return "", "", ""

        data = r.json()
        first_item = data[0] if isinstance(data, list) and data else data
        region = first_item.get("region") or ""
        market_url = first_item.get("market_url_270") or ""

        if not market_url:
            print(f"   ⚠️ No market_url_270 in response for {listing_id}")
            return "", "", region

        parsed = urlparse(market_url)
        params = {}
        for part in parsed.query.split("&"):
            if "=" in part:
                k, v = part.split("=", 1)
                params[k] = v

        ne_lat = float(params["ne_lat"]) if "ne_lat" in params else None
        ne_lng = float(params["ne_lng"]) if "ne_lng" in params else None
        sw_lat = float(params["sw_lat"]) if "sw_lat" in params else None
        sw_lng = float(params["sw_lng"]) if "sw_lng" in params else None

        area = calculate_bbox_area_km2(ne_lat, ne_lng, sw_lat, sw_lng)
        if not area or area <= 0:
            print(f"   ⚠️ Invalid bbox area for {listing_id}: {area}")
            return "", "", region

        density = 270 / area
        return classify_market_density(density), str(round(density, 4)), region

    except Exception as e:
        print(f"   ⚠️ Density exception for {listing_id}: {str(e)}")
        return "", "", ""

In [8]:
# Setup
gc = gspread.service_account(filename=SERVICE_ACCOUNT_FILE)
ss = gc.open(SPREADSHEET_NAME)

# Updated headers - Email is now first column
listings_headers = [
    "email", "listing_id", "airbnb_url", "Title", "start_date",
    "provider_status", "provider_onboarded_at", "provider_offboarded_at",
    "last_fetched_date", "last_fetched_day_index",
    "day_1_rank", "day_15_rank", "day_28_rank",
    "Guest Count", "Guest Favourite", "Initial Review Score", "Initial Review Count",
    "Rank Page - D1", "Rank Status - D1",
    "Rank Change - D15", "Rank Change Page - D15", "Rank Change Status - D15",
    "New Reviews D1-15",
    "Rank Change - D28", "Rank Change Page - D28", "Rank Change Status - D28",
    "New Reviews D15-28",
    "Market Density Class", "Market Density Value", "Region",
    "last_error", "updated_at"
]
raw_headers = ["listing_id", "rank_date", "day_index", "rank", "fetched_at"]
reviews_headers = ["listing_id", "review_date", "rating", "comment", "tag", "fetched_at"]


ws_ids = ss.worksheet(TAB_IDS)
ws_list = ensure_worksheet(ss, TAB_LISTINGS, listings_headers)
ws_raw = ensure_worksheet(ss, TAB_RAW, raw_headers)
ws_reviews = ensure_worksheet(ss, TAB_REVIEWS, reviews_headers)

today = date.today()
# Load existing data
list_vals = ws_list.get_all_values()
df_list = pd.DataFrame(list_vals[1:], columns=list_vals[0]) if len(list_vals) > 1 else pd.DataFrame(columns=list_vals[0])

existing_listing_ids = set(df_list["listing_id"].astype(str).tolist()) if not df_list.empty else set()

# STEP 1: Add new listings from IDs tab

In [9]:
print("=== STEP 1: ADD NEW LISTINGS ===")
ids_vals = ws_ids.get_all_values()
if not ids_vals:
    raise ValueError("IDs tab is empty or unreadable.")

ids_header = ids_vals[0]

# Ensure "Processed" column exists
if "Processed" not in ids_header:
    new_col_num = len(ids_header) + 1
    ws_ids.update(values=[["Processed"]], range_name=rowcol_to_a1(1, new_col_num))
    ids_vals = ws_ids.get_all_values()
    ids_header = ids_vals[0]
    print("✅ Added 'Processed' column")

if "Processed_Notes" not in ids_header:
    new_col_num = len(ids_header) + 1
    ws_ids.update(values=[["Processed_Notes"]], range_name=rowcol_to_a1(1, new_col_num))
    ids_vals = ws_ids.get_all_values()
    ids_header = ids_vals[0]
    print("✅ Added 'Processed_Notes' column")

ids_rows = ids_vals[1:]
df_ids = pd.DataFrame(ids_rows, columns=ids_header)

EMAIL_COL = "Email"
DATE_COL = "Submission date"
PROCESSED_COL = "Processed"
NOTES_COL = "Processed_Notes"
URL_COLS = [c for c in df_ids.columns if "Airbnb Property URL" in c]

if not URL_COLS:
    raise ValueError("No columns found matching 'Airbnb Property URL' in IDs tab header.")


# -----------------------------
# Process rows
# -----------------------------
new_rows = []
rows_to_update = []  # (row_num, status, notes)

for idx, r in df_ids.iterrows():
    current_status = (r.get(PROCESSED_COL, "") or "").strip().upper()

    # Skip already finalized rows (you can include ERROR here if you don't want retries)
    if current_status in {"YES", "INVALID_URL", "PARTIAL"}:
        continue

    email = (r.get(EMAIL_COL, "") or "").strip()
    start_date = to_date_str(r.get(DATE_COL))
    ids_row_num = idx + 2  # +2: header row + 0-index

    valid_count = 0
    invalid_count = 0
    added_count = 0
    existed_count = 0
    invalid_types = {}

    try:
        for url_col in URL_COLS:
            url = (r.get(url_col) or "").strip()
            if not url:
                continue

            listing_id = parse_airbnb_id_strict(url)

            if not listing_id:
                invalid_count += 1
                t = classify_input(url)
                invalid_types[t] = invalid_types.get(t, 0) + 1
                continue

            # Valid Airbnb listing link
            valid_count += 1

            if listing_id not in existing_listing_ids:
                new_rows.append([
    email, listing_id, url, "",   # A-D: Title left blank
    start_date,                    # E
    "NEW", "", "",                 # F=provider_status, G=provider_onboarded_at, H=provider_offboarded_at
    "", "0",                       # I=last_fetched_date, J=last_fetched_day_index
    "", "", "",                    # K=day_1_rank, L=day_15_rank, M=day_28_rank
    "", "", "", "",                # N=Guest Count, O=Guest Favourite, P=Initial Review Score, Q=Initial Review Count
    "", "",                        # R=Rank Page - D1, S=Rank Status - D1
    "", "", "",                    # T=Rank Change - D15, U=Rank Change Page - D15, V=Rank Change Status - D15
    "",                            # W=New Reviews D1-15
    "", "", "",                    # X=Rank Change - D28, Y=Rank Change Page - D28, Z=Rank Change Status - D28
    "",                            # AA=New Reviews D15-28
    "", "", "",                    # AB=Market Density Class, AC=Market Density Value, AD=Region
    "", now_iso()                  # AE=last_error, AF=updated_at
])
                existing_listing_ids.add(listing_id)
                added_count += 1
                print(f"➕ NEW: {listing_id} from {email}")
            else:
                existed_count += 1
                print(f"⏭️ SKIP: {listing_id} already exists")

        # Decide status for this IDs row
        if valid_count == 0 and invalid_count > 0:
            status = "INVALID_URL"
        elif valid_count > 0 and invalid_count > 0:
            status = "PARTIAL"
        elif valid_count > 0 and invalid_count == 0:
            status = "YES"
        else:
            # No URLs filled in at all (treat as INVALID_URL for CSM visibility)
            status = "INVALID_URL"
            invalid_count = max(invalid_count, 1)
            invalid_types["text"] = invalid_types.get("text", 0) + 1

        notes = build_csm_note(valid_count, invalid_count, invalid_types, status, added_count, existed_count)
        rows_to_update.append((ids_row_num, status, notes))

    except Exception as e:
        # Mark ERROR so CSM can ignore and you can fix automation
        rows_to_update.append((ids_row_num, "ERROR", f"{type(e).__name__}: {str(e)[:120]}"))


# -----------------------------
# Append new listings to Listings tab
# -----------------------------
if new_rows:
    ws_list.append_rows(new_rows, value_input_option="USER_ENTERED")
    print(f"✅ Added {len(new_rows)} new listings to Listings tab")
else:
    print("ℹ️ No new listings to add")


# -----------------------------
# Update Processed + Notes in IDs tab
# -----------------------------
processed_col_idx = ids_header.index("Processed")
notes_col_idx = ids_header.index("Processed_Notes")

# gspread uses 1-based indexing for row/col
start_row = 2  # data starts after header
end_row = len(ids_vals)

# Build full column update arrays for all existing data rows
processed_updates = []
notes_updates = []

# Create a map: row_num -> (status, notes)
updates_map = {row_num: (status, notes) for row_num, status, notes in rows_to_update}

for row_num in range(start_row, end_row + 1):
    if row_num in updates_map:
        status, notes = updates_map[row_num]
    else:
        # keep existing values if row was not processed
        existing_status = df_ids.iloc[row_num - 2].get(PROCESSED_COL, "")
        existing_notes = df_ids.iloc[row_num - 2].get(NOTES_COL, "")
        status = existing_status
        notes = existing_notes

    processed_updates.append([status])
    notes_updates.append([notes])

# Update both columns in only 2 requests instead of hundreds
processed_col_num = processed_col_idx + 1
notes_col_num = notes_col_idx + 1

processed_range = f"{rowcol_to_a1(2, processed_col_num)}:{rowcol_to_a1(end_row, processed_col_num)}"
notes_range = f"{rowcol_to_a1(2, notes_col_num)}:{rowcol_to_a1(end_row, notes_col_num)}"

ws_ids.batch_update([
    {"range": processed_range, "values": processed_updates},
    {"range": notes_range, "values": notes_updates},
])

print(f"✅ Updated {len(rows_to_update)} IDs row(s): statuses written (YES/INVALID_URL/PARTIAL/ERROR)")
# -----------------------------
# Refresh Listings DataFrame
# -----------------------------
list_vals = ws_list.get_all_values()
df_list = (
    pd.DataFrame(list_vals[1:], columns=list_vals[0])
    if len(list_vals) > 1
    else pd.DataFrame(columns=list_vals[0] if list_vals else [])
)

=== STEP 1: ADD NEW LISTINGS ===
➕ NEW: 1107991443646385508 from brett@portoro.com
➕ NEW: 1081869933681265232 from brett@portoro.com
➕ NEW: 1282788302076938003 from brett@portoro.com
➕ NEW: 1278939533299907631 from brett@portoro.com
➕ NEW: 1282786289120320946 from brett@portoro.com
✅ Added 5 new listings to Listings tab
✅ Updated 1 IDs row(s): statuses written (YES/INVALID_URL/PARTIAL/ERROR)


## STEP 2: Onboard NEW listings

In [10]:
print("\n=== STEP 2: ONBOARD NEW LISTINGS ===")

new_onboards = 0
listing_updates = []

for i in range(len(df_list)):
    if str(df_list.loc[i, "provider_status"]).strip() != "NEW":
        continue

    new_onboards += 1
    listing_id = str(df_list.loc[i, "listing_id"]).strip()
    row_num = i + 2
    ts = now_iso()

    code, txt = provider_onboard(listing_id)

    if code in (200, 201):
        listing_updates.extend([
            {"range": f"F{row_num}", "values": [["ONBOARDED"]]},
            {"range": f"G{row_num}", "values": [[ts]]},
            {"range": f"AE{row_num}", "values": [[""]]},
            {"range": f"AF{row_num}", "values": [[ts]]},
        ])
        print(f"✅ ONBOARDED {listing_id}")
    else:
        listing_updates.extend([
            {"range": f"F{row_num}", "values": [["ERROR"]]},
            {"range": f"AE{row_num}", "values": [[f"ONBOARD_FAIL: {txt[:300]}"]]},
            {"range": f"AF{row_num}", "values": [[ts]]},
        ])
        print(f"❌ ONBOARD FAIL {listing_id}: {txt[:200]}")

safe_batch_update(ws_list, listing_updates, chunk_size=100)

if new_onboards == 0:
    print("ℹ️ No new listings to onboard")
else:
    print(f"✅ Onboarding updates written for {new_onboards} listing(s)")


=== STEP 2: ONBOARD NEW LISTINGS ===
✅ ONBOARDED 1107991443646385508
✅ ONBOARDED 1081869933681265232
❌ ONBOARD FAIL 1282788302076938003: {"detail":"(1062, \"Duplicate entry '1282788302076938003-0' for key 'input_parameters.target_listing_id'\")"}
✅ ONBOARDED 1278939533299907631
❌ ONBOARD FAIL 1282786289120320946: {"detail":"(1062, \"Duplicate entry '1282786289120320946-0' for key 'input_parameters.target_listing_id'\")"}
✅ Onboarding updates written for 5 listing(s)


In [11]:
print("\n=== STEP 2.5: FETCH MISSING CONTENT ===")

content_fetches = 0
content_density_updates = []

for i in range(len(df_list)):
    if str(df_list.loc[i, "provider_status"]).strip() != "ONBOARDED":
        continue

    listing_id = str(df_list.loc[i, "listing_id"]).strip()
    row_num = i + 2

    guest_count = df_list.loc[i, "Guest Count"]
    density_filled = df_list.loc[i, "Market Density Class"]
    region_filled = df_list.loc[i, "Region"]
    gf = df_list.loc[i, "Guest Favourite"]
    review_score = df_list.loc[i, "Initial Review Score"]
    review_count = df_list.loc[i, "Initial Review Count"]
    title_existing = df_list.loc[i, "Title"]

    needs_content = any(
        not str(v).strip()
        for v in [guest_count, gf, review_score, review_count, title_existing]
    )

    needs_density = not str(density_filled).strip()
    needs_region = not str(region_filled).strip()

    if not needs_content and not needs_density and not needs_region:
        continue

    # -------------------------
    # CONTENT
    # -------------------------
    if needs_content:
        content_fetches += 1
        print(f"🔄 Fetching content for {listing_id}...")
        content_code, content_payload = provider_get_content(listing_id)

        if content_code == 200:
            try:
                content_data = content_payload[0] if isinstance(content_payload, list) else content_payload

                listing_title = str(content_data.get("listing_title", "") or "").strip()
                guest_cnt = parse_guest_count(content_data.get("max_guests", ""))
                guest_fav = "Yes" if content_data.get("guest_favorite", False) else "No"
                rating = str(content_data.get("overall_rating", "") or "")
                review_cnt = str(len(content_data.get("reviews", [])))
                ts = now_iso()

                updates = [
                    {"range": f"N{row_num}", "values": [[guest_cnt]]},
                    {"range": f"O{row_num}", "values": [[guest_fav]]},
                    {"range": f"P{row_num}", "values": [[rating]]},
                    {"range": f"Q{row_num}", "values": [[review_cnt]]},
                    {"range": f"AF{row_num}", "values": [[ts]]},
                ]

                if listing_title and not str(title_existing).strip():
                    updates.append({
                        "range": f"D{row_num}",
                        "values": [[listing_title]]
                    })

                if not guest_cnt:
                    updates.append({
                        "range": f"AE{row_num}",
                        "values": [[f"CONTENT_INCOMPLETE: No guest count (will retry)"]]
                    })
                    print(f"⚠️ Content incomplete for {listing_id}: No guest count")
                else:
                    updates.append({
                        "range": f"AE{row_num}",
                        "values": [[""]]
                    })

                content_density_updates.extend(updates)

                print(
                    f"✅ CONTENT {listing_id} | Title:{listing_title[:50]} | "
                    f"Guests:{guest_cnt} Fav:{guest_fav} Rating:{rating} Reviews:{review_cnt}"
                )

            except Exception as e:
                content_density_updates.extend([
                    {"range": f"AE{row_num}", "values": [[f"CONTENT_ERROR: {str(e)[:200]}"]]},
                    {"range": f"AF{row_num}", "values": [[now_iso()]]},
                ])
                print(f"⚠️ Content parse error for {listing_id}: {str(e)[:100]}")
        else:
            content_density_updates.extend([
                {"range": f"AE{row_num}", "values": [[f"CONTENT_FETCH_FAIL: Code {content_code} (will retry)"]]},
                {"range": f"AF{row_num}", "values": [[now_iso()]]},
            ])
            print(f"⚠️ Content fetch failed for {listing_id}, code: {content_code}")

    # -------------------------
    # DENSITY
    # -------------------------
    if needs_density or needs_region:
        print(f"🗺️ Fetching density for {listing_id}...")
        density_class, density_value, region = get_market_density(listing_id)

        if density_class or region:
            density_updates = []

            if density_class and needs_density:
                density_updates.append({"range": f"AB{row_num}", "values": [[density_class]]})
                density_updates.append({"range": f"AC{row_num}", "values": [[density_value]]})

            if region and needs_region:
                density_updates.append({"range": f"AD{row_num}", "values": [[region]]})

            if density_updates:
                density_updates.append({"range": f"AF{row_num}", "values": [[now_iso()]]})
                content_density_updates.extend(density_updates)

            print(f"✅ DENSITY {listing_id} | Class:{density_class} Value:{density_value} Region:{region}")
        else:
            print(f"⚠️ Density fetch failed or no area data for {listing_id}")

safe_batch_update(ws_list, content_density_updates, chunk_size=100)

if content_fetches == 0:
    print("ℹ️ No content fetches needed")
else:
    print(f"🔄 Attempted {content_fetches} content fetches")

# Refresh dataframe
list_vals = ws_list.get_all_values()
df_list = pd.DataFrame(list_vals[1:], columns=list_vals[0])


=== STEP 2.5: FETCH MISSING CONTENT ===
🔄 Fetching content for 1494760145387157393...
⚠️ Content parse error for 1494760145387157393: list index out of range
🔄 Fetching content for 1635728237696006202...
✅ CONTENT 1635728237696006202 | Title:6k SqFt Estate -  Epic Amenities & Fun - Sleeps 22 | Guests:16 Fav:No Rating: Reviews:2
🔄 Fetching content for 1598644098338980295...
✅ CONTENT 1598644098338980295 | Title:Beach Walker Villa with Ocean Views & Large Pool! | Guests:6 Fav:No Rating: Reviews:0
🔄 Fetching content for 1567077024886979877...
✅ CONTENT 1567077024886979877 | Title:NEW! 4BR | Pool | Sleeps 12 | 6 miles to AMI | Guests:12 Fav:No Rating: Reviews:1
🔄 Fetching content for 1537943328629978746...
✅ CONTENT 1537943328629978746 | Title:Salt + Sunset by Stay on 30a | Guests:8 Fav:No Rating: Reviews:1
🔄 Fetching content for 1369661870630530055...
✅ CONTENT 1369661870630530055 | Title:Three Duval Street Suites w parking and pool | Guests:12 Fav:No Rating: Reviews:2
🔄 Fetching content

## STEP 3: Daily rank tracking + content fetch on day 0

In [12]:
print("\n=== STEP 3: RANK TRACKING (30-DAY BACKFILL, FIXED DEDUP) ===")

# -------------------------
# HARDCODED RUN DATE
# -------------------------
from datetime import date, datetime, timedelta
today = date.today()

# -------------------------
# HELPERS
# -------------------------
def clean_cell(val):
    if pd.isna(val):
        return ""
    s = str(val).strip()
    return "" if s.lower() == "nan" else s

def parse_sheet_date(date_str):
    date_str = clean_cell(date_str)
    if not date_str:
        return None
    try:
        return datetime.strptime(date_str[:10], "%Y-%m-%d").date()
    except ValueError:
        try:
            return datetime.strptime(date_str, "%d/%m/%Y").date()
        except ValueError:
            return None

def extract_rank_from_payload(payload):
    rank_val = None
    if isinstance(payload, list) and payload:
        rank_val = payload[0].get("listed_rank")
        if rank_val is None:
            rank_val = payload[0].get("listedrank")
    elif isinstance(payload, dict):
        rank_val = payload.get("listed_rank") or payload.get("listedrank") or payload.get("rank")

    if rank_val is None:
        return None

    rank_str = str(rank_val).strip()
    if not rank_str:
        return None

    return rank_str

# -------------------------
# LOAD EXISTING RAW DATA
# -------------------------
raw_vals = ws_raw.get_all_values()

if len(raw_vals) > 1:
    df_raw = pd.DataFrame(raw_vals[1:], columns=raw_vals[0])

    df_raw.columns = [str(c).strip() for c in df_raw.columns]
    raw_cols = {c: c for c in df_raw.columns}

    listing_col = None
    rank_date_col = None

    for c in ["listing_id", "listingid"]:
        if c in raw_cols:
            listing_col = raw_cols[c]
            break

    for c in ["rank_date", "rankdate"]:
        if c in raw_cols:
            rank_date_col = raw_cols[c]
            break

    if listing_col:
        df_raw[listing_col] = df_raw[listing_col].astype(str).str.strip()
    if rank_date_col:
        df_raw[rank_date_col] = df_raw[rank_date_col].astype(str).str.strip()

    today_str = today.isoformat()

    if listing_col and rank_date_col:
        existing_raw_keys = set(
            (
                df_raw[listing_col] + "|" + df_raw[rank_date_col]
            ).tolist()
        )
        df_raw_today = df_raw[df_raw[rank_date_col] == today_str].copy()
    else:
        existing_raw_keys = set()
        df_raw_today = pd.DataFrame()
        print("⚠️ Could not find raw sheet columns for deduping")

    print("📊 Data_Raw current state:")
    print(f"   - Total historical rows: {len(df_raw)}")
    print(f"   - Today's fetches: {len(df_raw_today)}")
    print(f"   - Checking against {len(existing_raw_keys)} existing listing/date keys")
    print(f"   - Using columns: listing={listing_col}, rank_date={rank_date_col}")
else:
    df_raw = pd.DataFrame()
    df_raw_today = pd.DataFrame()
    existing_raw_keys = set()
    print("📊 Data_Raw is empty (new sheet)")

raw_append = []
rank_listing_updates = []

for i in range(len(df_list)):
    provider_status = clean_cell(df_list.loc[i, "provider_status"])
    if provider_status != "ONBOARDED":
        continue

    listing_id = clean_cell(df_list.loc[i, "listing_id"])
    start_date_raw = df_list.loc[i, "start_date"]
    start_date = clean_cell(start_date_raw)
    density_filled = clean_cell(df_list.loc[i, "Market Density Class"])
    region_filled = clean_cell(df_list.loc[i, "Region"])
    row_num = i + 2

    if not listing_id:
        print(f"⚠️ Skipping row {row_num} due to empty listing_id")
        continue

    if not start_date:
        print(f"⚠️ Skipping {listing_id} due to empty start_date")
        continue

    sd = parse_sheet_date(start_date)
    if sd is None:
        print(f"❌ Invalid date format for {listing_id}: {start_date}")
        continue

    current_day_index = (today - sd).days

    if current_day_index < 0:
        print(f"⚠️ Skipping {listing_id} because start_date is in the future")
        continue

    # -------------------------
    # TODAY DAILY TRACKING
    # -------------------------
    rank_date_today = today.isoformat()
    key_today = f"{listing_id}|{rank_date_today}"

    if key_today in existing_raw_keys:
        print(f"⏭️ SKIP {listing_id} — rank already exists for {rank_date_today}")
    else:
        code_today, payload_today = provider_get_rank(listing_id, rank_date_today)
        fetched_at_today = now_iso()

        if code_today == 200:
            rank_val_today = extract_rank_from_payload(payload_today)

            if rank_val_today is not None:
                raw_append.append([
                    str(listing_id),
                    rank_date_today,
                    str(current_day_index),
                    rank_val_today,
                    fetched_at_today
                ])
                existing_raw_keys.add(key_today)

                if current_day_index == 0:
                    rank_listing_updates.append({"range": f"K{row_num}", "values": [[rank_val_today]]})
                elif current_day_index == 15:
                    rank_listing_updates.append({"range": f"L{row_num}", "values": [[rank_val_today]]})
                elif current_day_index == 28:
                    rank_listing_updates.append({"range": f"M{row_num}", "values": [[rank_val_today]]})

            rank_listing_updates.extend([
                {"range": f"I{row_num}", "values": [[rank_date_today]]},
                {"range": f"J{row_num}", "values": [[str(current_day_index)]]},
                {"range": f"AF{row_num}", "values": [[fetched_at_today]]},
            ])

            print(f"📈 {listing_id} Day {current_day_index} rank={rank_val_today}")
        else:
            rank_listing_updates.extend([
                {"range": f"AE{row_num}", "values": [[f"RANK_FAIL({rank_date_today}): {str(payload_today)[:300]}"]]},
                {"range": f"AF{row_num}", "values": [[fetched_at_today]]},
            ])
            print(f"❌ RANK FAIL {listing_id} Day {current_day_index}: {str(payload_today)[:200]}")

    # -------------------------
    # BACKFILL: days 0 to current_day_index, capped at 28
    # -------------------------
    print(f"🔄 Backfilling missing days for {listing_id}...")
    backfill_count = 0

    for days_ago in range(1, current_day_index + 1):
        backfill_date_obj = today - timedelta(days=days_ago)
        backfill_date = backfill_date_obj.isoformat()
        backfill_key = f"{listing_id}|{backfill_date}"

        if backfill_key in existing_raw_keys:
            print(f"  ⏭️ SKIP {listing_id} — rank already exists for {backfill_date}")
            continue

        actual_day_index = (backfill_date_obj - sd).days

        if actual_day_index < 0:
            continue

        code, payload = provider_get_rank(listing_id, backfill_date)
        fetched_at = now_iso()

        if code == 200:
            rank_val = extract_rank_from_payload(payload)

            if rank_val is not None:
                raw_append.append([
                    str(listing_id),
                    backfill_date,
                    str(actual_day_index),
                    rank_val,
                    fetched_at
                ])
                existing_raw_keys.add(backfill_key)
                backfill_count += 1

                if actual_day_index == 0:
                    rank_listing_updates.append({"range": f"K{row_num}", "values": [[rank_val]]})
                elif actual_day_index == 15:
                    rank_listing_updates.append({"range": f"L{row_num}", "values": [[rank_val]]})
                elif actual_day_index == 28:
                    rank_listing_updates.append({"range": f"M{row_num}", "values": [[rank_val]]})

                print(f"  📊 {listing_id} day{actual_day_index:2d} ({backfill_date}): {rank_val}")
        else:
            print(f"  ⚠️ Backfill failed for {listing_id} on {backfill_date}: {str(payload)[:200]}")

        time.sleep(0.1)

    print(f"  ✅ Backfilled {backfill_count} new days for {listing_id}")

    # -------------------------
    # DENSITY + REGION FETCH IF MISSING
    # -------------------------
    needs_density = not density_filled
    needs_region = not region_filled

    if current_day_index == 0 or needs_density or needs_region:
        print(f"🗺️ Fetching density for {listing_id}...")
        density_class, density_value, region = get_market_density(listing_id)

        if density_class or region:
            density_updates = []

            if density_class and needs_density:
                density_updates.append({"range": f"AB{row_num}", "values": [[density_class]]})
                density_updates.append({"range": f"AC{row_num}", "values": [[density_value]]})

            if region and needs_region:
                density_updates.append({"range": f"AD{row_num}", "values": [[region]]})

            if density_updates:
                density_updates.append({"range": f"AF{row_num}", "values": [[now_iso()]]})
                rank_listing_updates.extend(density_updates)

            print(f"✅ DENSITY {listing_id} | Class:{density_class} Value:{density_value} Region:{region}")
        else:
            print(f"⚠️ Density fetch failed or no area data for {listing_id}")

# -------------------------
# DEDUP RAW APPEND IN-MEMORY
# -------------------------
deduped_raw_append = []
seen_raw_rows = set()

for row in raw_append:
    row_key = f"{str(row[0]).strip()}|{str(row[1]).strip()}"
    if row_key not in seen_raw_rows:
        seen_raw_rows.add(row_key)
        deduped_raw_append.append(row)

raw_append = deduped_raw_append

# -------------------------
# WRITE LISTINGS UPDATES SAFELY
# -------------------------
safe_batch_update(ws_list, rank_listing_updates, chunk_size=100)

# -------------------------
# APPEND TO DATA_RAW SAFELY
# -------------------------
print("\n📝 Data_Raw append operation:")
print(f"   - Rows to append: {len(raw_append)}")

if raw_append:
    print(f"   - Sample row: {raw_append[0]}")
    try:
        rows_before = len(ws_raw.get_all_values())
        print(f"   - Rows before: {rows_before}")

        safe_append_rows(
            ws_raw,
            raw_append,
            value_input_option="USER_ENTERED",
            table_range="A1"
        )

        rows_after = len(ws_raw.get_all_values())
        print(f"   - Rows after: {rows_after}")
        print(f"   - New rows added: {rows_after - rows_before}")

        if rows_after - rows_before == len(raw_append):
            print(f"✅ Successfully appended {len(raw_append)} rows to Data_Raw")
        else:
            print(f"⚠️ WARNING: Expected +{len(raw_append)} rows, got +{rows_after - rows_before}")

    except Exception as e:
        print(f"❌ ERROR appending to Data_Raw: {str(e)}")
        import traceback
        print(traceback.format_exc())
else:
    print("ℹ️ No new ranks to append")

Streaming output truncated to the last 5000 lines.
  ⏭️ SKIP 1360299925560671495 — rank already exists for 2026-03-26
  ⏭️ SKIP 1360299925560671495 — rank already exists for 2026-03-25
  ✅ Backfilled 0 new days for 1360299925560671495
📈 1524042145831720679 Day 15 rank=91
🔄 Backfilling missing days for 1524042145831720679...
  ⏭️ SKIP 1524042145831720679 — rank already exists for 2026-04-08
  ⏭️ SKIP 1524042145831720679 — rank already exists for 2026-04-07
  ⏭️ SKIP 1524042145831720679 — rank already exists for 2026-04-06
  ⏭️ SKIP 1524042145831720679 — rank already exists for 2026-04-05
  ⏭️ SKIP 1524042145831720679 — rank already exists for 2026-04-04
  ⏭️ SKIP 1524042145831720679 — rank already exists for 2026-04-03
  ⏭️ SKIP 1524042145831720679 — rank already exists for 2026-04-02
  ⏭️ SKIP 1524042145831720679 — rank already exists for 2026-04-01
  ⏭️ SKIP 1524042145831720679 — rank already exists for 2026-03-31
  ⏭️ SKIP 1524042145831720679 — rank already exists for 2026-03-30
  ⏭️

In [19]:
print("\n=== STEP 4: REVIEWS RAW ===")

# ── 1. Fresh sheet read ───────────────────────────────────────────────────────
reviews_vals = ws_reviews.get_all_values()
existing_review_keys = set()

if len(reviews_vals) > 1:
    headers = [str(c).strip() for c in reviews_vals[0]]
    print(f"📋 Sheet headers : {headers}")
    print(f"📋 Total rows    : {len(reviews_vals) - 1}")

    # Find the correct listing_id column — try both 'listing_id' and 'Listing'
    li_col = None
    for candidate in ["listing_id", "Listing"]:
        if candidate in headers:
            li_col = headers.index(candidate)
            break

    rd_col = headers.index("review_date") if "review_date" in headers else None

    if li_col is None or rd_col is None:
        print(f"⚠️ Could not find required columns (li_col={li_col}, rd_col={rd_col}). Skipping dedup.")
        existing_review_keys = set()
    else:
        for row in reviews_vals[1:]:
            row = row + [""] * (max(li_col, rd_col) + 1)
            raw_id = row[li_col].strip()
            if not raw_id or raw_id.lower() in ("listing_id", "listing"):
                continue
            try:
                raw_id = str(int(float(raw_id)))
            except Exception:
                pass
            rdate = row[rd_col].strip()
            existing_review_keys.add(f"{raw_id}|{rdate}")

    print(f"📋 Loaded {len(existing_review_keys)} dedup keys")
    print(f"   Sample : {next(iter(existing_review_keys), None)}")
else:
    print("📋 Reviews Raw sheet is empty")

# ── 2. Collect new reviews ────────────────────────────────────────────────────
reviews_append = []

for i in range(len(df_list)):
    if str(df_list.loc[i, "provider_status"]).strip() != "ONBOARDED":
        continue

    listing_id = str(df_list.loc[i, "listing_id"]).strip()
    start_date = str(df_list.loc[i, "start_date"]).strip()

    if not start_date:
        continue

    parsed_date = None
    for fmt in ("%Y-%m-%d", "%d/%m/%Y"):
        try:
            parsed_date = datetime.strptime(start_date[:10], fmt)
            break
        except ValueError:
            continue
    if not parsed_date:
        print(f"⚠️ Invalid date for {listing_id}: {start_date}")
        continue

    # Normalize listing_id to avoid float-format mismatches
    try:
        norm_id = str(int(float(listing_id)))
    except Exception:
        norm_id = listing_id

    content_code, content_payload = provider_get_content(listing_id)
    if content_code != 200:
        print(f"⚠️ Reviews fetch failed for {listing_id}, code: {content_code}")
        continue

    try:
        content_data = (
            content_payload[0] if isinstance(content_payload, list) and content_payload
            else content_payload if isinstance(content_payload, dict)
            else None
        )
        if not content_data:
            print(f"⚠️ Empty content payload for {listing_id}, skipping")
            continue

        reviews    = content_data.get("reviews", [])
        fetched_at = now_iso()
        new_count  = 0

        for review in reviews:
            ts      = str(review.get("timestamp", "") or "").strip()[:10]
            comment = str(review.get("comment",   "") or "").strip()
            rating  = str(review.get("rating",    "") or "").strip()
            tag     = str(review.get("tag",       "") or "")

            if not ts:
                continue

            key = f"{norm_id}|{ts}"
            if key in existing_review_keys:
                continue

            reviews_append.append([norm_id, ts, rating, comment, tag, fetched_at])
            existing_review_keys.add(key)
            new_count += 1

        print(f"✅ {listing_id}: {len(reviews)} reviews fetched, {new_count} new")

    except Exception as e:
        print(f"⚠️ Reviews parse error for {listing_id}: {str(e)[:100]}")

# ── 3. Append ─────────────────────────────────────────────────────────────────
print(f"\n📝 Rows to append: {len(reviews_append)}")

if reviews_append:
    try:
        safe_append_rows(ws_reviews, reviews_append,
                         value_input_option="USER_ENTERED", table_range="A1")
        print(f"✅ Appended {len(reviews_append)} reviews to Reviews_Raw")
    except Exception as e:
        print(f"❌ ERROR appending to Reviews_Raw: {str(e)}")
else:
    print("ℹ️ No new reviews to append")


=== STEP 4: REVIEWS RAW ===
📋 Sheet headers : ['listing_id', 'review_date', 'rating', 'comment', 'tag', 'fetched_at']
📋 Total rows    : 51329
📋 Loaded 51329 dedup keys
   Sample : 11387629|2024-04-23
⚠️ Empty content payload for 1494760145387157393, skipping
✅ 1336199992686900402: 44 reviews fetched, 0 new
✅ 1086222414975552288: 48 reviews fetched, 0 new
✅ 1635728237696006202: 2 reviews fetched, 0 new
✅ 1329278162205669599: 30 reviews fetched, 0 new
✅ 1292957343666552810: 55 reviews fetched, 0 new
✅ 49294430: 35 reviews fetched, 0 new
✅ 51861236: 5 reviews fetched, 0 new
✅ 46194666: 23 reviews fetched, 0 new
✅ 1598644098338980295: 0 reviews fetched, 0 new
✅ 50500319: 18 reviews fetched, 0 new
✅ 1567077024886979877: 1 reviews fetched, 0 new
✅ 1391399021291310864: 29 reviews fetched, 0 new
✅ 1015315101971702374: 46 reviews fetched, 0 new
✅ 1481067760500680704: 4 reviews fetched, 0 new
✅ 1075144668717179067: 66 reviews fetched, 0 new
✅ 992939179438897760: 74 reviews fetched, 0 new
✅ 1538

In [18]:
print("\n=== ONE-TIME BACKFILL: day_28_rank from Data_Raw day 27 ===")

# ── 1. Load raw data safely ───────────────────────────────────────────────────
ws_raw_updated = ss.worksheet("Data Raw Updated")
raw_all_vals = ws_raw_updated.get_all_values()

if not raw_all_vals or len(raw_all_vals) <= 1:
    print("⚠️ Data Raw Updated sheet is empty, nothing to backfill")
else:
    df_raw_all = pd.DataFrame(raw_all_vals[1:], columns=raw_all_vals[0])

    # Normalize listing_id to avoid float-format mismatches e.g. 1234567.0 vs 1234567
    df_raw_all["listing_id"] = df_raw_all["listing_id"].apply(
        lambda x: str(int(float(x))) if str(x).strip() and str(x).strip().lower() not in ("nan", "none") else x
    )

    # ── 2. Build day 27 lookup ────────────────────────────────────────────────
    df_day27 = df_raw_all[df_raw_all["day_index"].astype(str).str.strip() == "27"]
    day27_lookup = dict(zip(df_day27["listing_id"].astype(str), df_day27["rank"].astype(str)))
    print(f"📊 Found {len(day27_lookup)} listings with day 27 rank data")

    # ── 3. Build updates ──────────────────────────────────────────────────────
    backfill_updates = []

    for i in range(len(df_list)):
        listing_id = str(df_list.loc[i, "listing_id"]).strip()
        current_day28_rank = str(df_list.loc[i, "day_28_rank"]).strip()
        row_num = i + 2

        # Normalize listing_id
        try:
            norm_id = str(int(float(listing_id)))
        except Exception:
            norm_id = listing_id

        # Only backfill if day_28_rank is genuinely empty
        if not current_day28_rank or current_day28_rank.lower() in ("nan", "none"):
            day27_val = day27_lookup.get(norm_id, "")

            if day27_val and day27_val.lower() not in ("nan", "none", ""):
                backfill_updates.append({
                    "range": f"M{row_num}",
                    "values": [[day27_val]]
                })
                print(f"🔁 {norm_id} → backfilled day_28_rank with day 27 rank={day27_val}")
            else:
                print(f"⚠️ {norm_id} → no day 27 rank found in Data Raw Updated, skipping")

    # ── 4. Write updates ──────────────────────────────────────────────────────
    if backfill_updates:
        safe_batch_update(ws_list, backfill_updates, chunk_size=100)
        print(f"✅ Done. Backfilled {len(backfill_updates)} rows")
    else:
        print("ℹ️ No rows needed backfilling")


=== ONE-TIME BACKFILL: day_28_rank from Data_Raw day 27 ===
📊 Found 252 listings with day 27 rank data
🔁 1336199992686900480 → backfilled day_28_rank with day 27 rank=41
🔁 1086222414975552256 → backfilled day_28_rank with day 27 rank=25
🔁 1635728237696006144 → backfilled day_28_rank with day 27 rank=3
🔁 1329278162205669632 → backfilled day_28_rank with day 27 rank=84
🔁 1292957343666552832 → backfilled day_28_rank with day 27 rank=7
🔁 49294430 → backfilled day_28_rank with day 27 rank=203
🔁 51861236 → backfilled day_28_rank with day 27 rank=236
🔁 46194666 → backfilled day_28_rank with day 27 rank=101
🔁 1598644098338980352 → backfilled day_28_rank with day 27 rank=58
🔁 50500319 → backfilled day_28_rank with day 27 rank=211
🔁 1567077024886979840 → backfilled day_28_rank with day 27 rank=120
🔁 1391399021291310848 → backfilled day_28_rank with day 27 rank=44
🔁 1015315101971702400 → backfilled day_28_rank with day 27 rank=70
🔁 1481067760500680704 → backfilled day_28_rank with day 27 rank=92